# Transcrição do vídeo — Seminário I

1. Extrai o áudio do vídeo com `ffmpeg` (WAV mono 16 kHz, o formato que o Whisper usa internamente).
2. Transcreve com `faster-whisper` (CPU), em português.

## 1. Configuração

In [1]:
from pathlib import Path

PASTA = Path.cwd()  # 07_revisoes_banca/seminario_I
RAIZ = PASTA.parents[1]  # rev-sist-dissertacao
VIDEO = RAIZ / "arquivos-dissertacao" / "video.mp4"

PASTA_AUDIO = PASTA / "audio"
PASTA_AUDIO.mkdir(exist_ok=True)
AUDIO = PASTA_AUDIO / (VIDEO.stem + ".wav")

SAIDA = PASTA / "transcricao"
SAIDA.mkdir(exist_ok=True)

# Modelo por dispositivo: na GPU cabe o large-v3; na CPU, medium para não demorar horas.
MODELO_GPU = "large-v3"  # tiny | base | small | medium | large-v3
MODELO_CPU = "medium"
IDIOMA = "pt"

print(VIDEO, VIDEO.exists())

/home/silvio/Projetos/rev-sist-dissertacao/arquivos-dissertacao/video.mp4 True


## 2. Extrair o áudio do vídeo

Pula a extração se o `.wav` já existir (útil para rodar de novo só a transcrição).

In [2]:
import shutil
import subprocess

assert shutil.which("ffmpeg"), "ffmpeg não encontrado no PATH"
assert VIDEO.exists(), f"Vídeo não encontrado: {VIDEO}"

if AUDIO.exists():
    print("Áudio já extraído:", AUDIO)
else:
    subprocess.run(
        [
            "ffmpeg", "-hide_banner", "-loglevel", "error",
            "-i", str(VIDEO),
            "-vn",              # descarta o vídeo
            "-ac", "1",         # mono
            "-ar", "16000",     # 16 kHz
            "-c:a", "pcm_s16le",
            str(AUDIO),
        ],
        check=True,
    )
    print("Áudio salvo em", AUDIO)

print(f"{AUDIO.stat().st_size / 1e6:.1f} MB")

Áudio já extraído: /home/silvio/Projetos/rev-sist-dissertacao/07_revisoes_banca/seminario_I/audio/video.wav
106.1 MB


## 3. Detectar GPU e carregar o modelo

Tenta CUDA (float16); se não houver placa ou o carregamento falhar (driver, cuBLAS/cuDNN), cai para CPU (int8).

In [3]:
import ctypes, glob, os

# O ctranslate2 não procura cuBLAS/cuDNN dentro do venv: carrega as libs dos pacotes
# nvidia-cublas-cu12 / nvidia-cudnn-cu12 antes do modelo. Sem eles, segue para CPU.
try:
    import nvidia.cublas, nvidia.cudnn
    for pkg, libs in [(nvidia.cublas, ["libcublasLt.so.12", "libcublas.so.12"]),
                      (nvidia.cudnn, ["libcudnn*.so.9"])]:
        d = os.path.join(list(pkg.__path__)[0], "lib")
        for pat in libs:
            for f in sorted(glob.glob(os.path.join(d, pat))):
                ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
except (ImportError, OSError) as e:
    print("Libs CUDA não carregadas:", e)

import ctranslate2
from faster_whisper import WhisperModel

model = None
if ctranslate2.get_cuda_device_count() > 0:
    try:
        model = WhisperModel(MODELO_GPU, device="cuda", compute_type="float16")
        DISPOSITIVO, MODELO = "cuda", MODELO_GPU
    except Exception as e:
        print("GPU detectada, mas falhou ao carregar o modelo:", e)

if model is None:
    model = WhisperModel(MODELO_CPU, device="cpu", compute_type="int8")
    DISPOSITIVO, MODELO = "cpu", MODELO_CPU

print(f"Dispositivo: {DISPOSITIVO} | modelo: {MODELO}")

Dispositivo: cuda | modelo: large-v3


## 4. Transcrever

O WAV é lido direto em numpy: o faster-whisper 1.2.1 quebra ao abrir arquivo com o PyAV 19 instalado (`metadata_errors`).

In [4]:
import wave
import numpy as np

with wave.open(str(AUDIO)) as w:
    assert w.getframerate() == 16000 and w.getnchannels() == 1 and w.getsampwidth() == 2
    audio = np.frombuffer(w.readframes(w.getnframes()), np.int16).astype(np.float32) / 32768

segments, info = model.transcribe(
    audio,
    language=IDIOMA,
    vad_filter=True,
    beam_size=5,
)
print(f"Duração: {info.duration/60:.1f} min")

segmentos = []
for s in segments:
    segmentos.append(s)
    print(f"[{s.start:7.1f}s] {s.text.strip()}")

Duração: 55.2 min
[    1.7s] da apresentação. Então vamos lá, senhores, dando continuidade ao trabalho do Silvio Ferreira Gomes Júnior,
[    7.7s] aluno do mestrado do PPCA, hoje nós vamos fazer o seminário 1 do trabalho, cujo título é
[   13.5s] Uma Proposta de Arquitetura de Lake House para o Exército Brasileiro no Contexto de Comando e Controle.
[   19.6s] Então hoje é 18 de novembro, são 4 minutos para as 3 horas.
[   25.6s] Então, mais uma vez, agradeço a colaboração dos professores,
[   30.0s] as dicas estão sendo muito boas, o Silvio está evoluindo o trabalho.
[   33.7s] O objetivo da reunião é basicamente isso, é o Silvio apresentar as sugestões,
[   37.7s] como é que elas estão sendo implementadas e o que falta para a próxima etapa até a defesa.
[   43.9s] Tem mais o seminário 2 e a defesa. Então ele vai dar um andamento, como é que foram feitas as alterações
[   49.1s] e o trabalho futuro para concluir o trabalho.
[   52.1s] Silvio, abre a apresentação aí, você tem 20 minutos

## 5. Salvar

In [5]:
def fmt(t):
    h, r = divmod(int(t), 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

base = SAIDA / AUDIO.stem
(base.with_suffix(".txt")).write_text(
    "\n".join(s.text.strip() for s in segmentos), encoding="utf-8")
(base.with_name(base.name + "_tempos.txt")).write_text(
    "\n".join(f"[{fmt(s.start)} - {fmt(s.end)}] {s.text.strip()}" for s in segmentos),
    encoding="utf-8")
print("Salvo em", SAIDA)

Salvo em /home/silvio/Projetos/rev-sist-dissertacao/07_revisoes_banca/seminario_I/transcricao


## 6. Diarização (quem fala quando) — pyannote na GPU

Separa as vozes pelo som, sem olhar o texto. Pré-requisitos (uma vez só):
1. token *Read* em https://huggingface.co/settings/tokens;
2. aceitar os termos em https://huggingface.co/pyannote/speaker-diarization-community-1;
3. rodar a célula de login abaixo (ou `uv run hf auth login` num terminal).

Independe das seções 3–5: relê a transcrição de `transcricao/<nome>_tempos.txt`. Rodar a seção 1 antes.

In [6]:
# Login no Hugging Face (só na primeira vez: o token fica salvo em ~/.cache/huggingface/).
# getpass abre uma caixa oculta: o token não aparece na tela nem fica gravado no notebook.
from getpass import getpass
from huggingface_hub import get_token, login

if get_token() is None:
    login(token=getpass("Token do Hugging Face: "), add_to_git_credential=False)
else:
    print("Já logado no Hugging Face.")

In [7]:
import wave
import numpy as np
import torch
from pyannote.audio import Pipeline

N_FALANTES = 4  # Victorino, Silvio, Maria Cláudia, Andrei

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=True)
pipeline.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# Passa o áudio já em memória (evita depender do decodificador de arquivos).
with wave.open(str(AUDIO)) as w:
    sr = w.getframerate()
    onda = np.frombuffer(w.readframes(w.getnframes()), np.int16).astype(np.float32) / 32768
saida = pipeline({"waveform": torch.from_numpy(onda)[None], "sample_rate": sr},
                 num_speakers=N_FALANTES)
diar = saida.exclusive_speaker_diarization  # sem sobreposição: um falante por instante

turnos = [(t.start, t.end, spk) for t, _, spk in diar.itertracks(yield_label=True)]
print(f"{len(turnos)} turnos, falantes: {sorted(diar.labels())}")

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/xvec_transform.npz: downloading bytes:           |  0.00B            

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

/home/silvio/Projetos/rev-sist-dissertacao/.venv/lib64/python3.14/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/home/silvio/Projetos/rev-sist-dissertacao/.venv/lib64/python3.14/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1927.)
  std = sequences.std(dim=-1, correction=1)


699 turnos, falantes: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03']


## 7. Dar nome aos falantes e cruzar com a transcrição

O pyannote devolve rótulos anônimos (`SPEAKER_00`…). O nome vem dos blocos da sessão: o
rótulo que mais fala dentro do bloco de cada pessoa é essa pessoa. Depois, cada linha do
Whisper recebe o falante com maior sobreposição de tempo — é isso que pega as
intervenções do Silvio no meio das considerações da banca.

In [8]:
import re
from collections import defaultdict

def seg(hms):
    h, m, s = map(int, hms.split(":"))
    return h * 3600 + m * 60 + s

# Blocos informados pelo Silvio (aproximados): (início, fim, quem conduz)
BLOCOS = [
    ("00:00:00", "00:01:25", "Victorino"),
    ("00:01:25", "00:22:30", "Silvio"),
    ("00:22:30", "00:23:21", "Victorino"),
    ("00:23:21", "00:37:30", "Maria Cláudia"),
    ("00:37:30", "00:38:56", "Victorino"),
    ("00:38:56", "99:00:00", "Andrei"),
]
BLOCOS = [(seg(a), seg(b), n) for a, b, n in BLOCOS]

def sobrepoe(a0, a1, b0, b1):
    return max(0.0, min(a1, b1) - max(a0, b0))

# Tempo de cada rótulo dentro dos blocos de cada pessoa
tempo = defaultdict(lambda: defaultdict(float))  # pessoa -> rótulo -> segundos
for b0, b1, pessoa in BLOCOS:
    for t0, t1, spk in turnos:
        tempo[pessoa][spk] += sobrepoe(b0, b1, t0, t1)

# Atribuição gulosa: a pessoa com o rótulo mais dominante escolhe primeiro
NOMES, usados = {}, set()
for pessoa, spk, s in sorted(((p, k, v) for p, d in tempo.items() for k, v in d.items()),
                             key=lambda x: -x[2]):
    if pessoa not in NOMES.values() and spk not in usados:
        NOMES[spk] = pessoa
        usados.add(spk)
for spk in diar.labels():
    NOMES.setdefault(spk, spk)  # sobrou rótulo sem dono: fica anônimo

for p in tempo:
    tot = sum(tempo[p].values()) or 1
    print(f"{p:14s}", {NOMES[k]: f"{v/tot:.0%}" for k, v in sorted(tempo[p].items(), key=lambda x: -x[1]) if v})

# Relê a transcrição com tempos (precisão de segundo) e atribui falante por sobreposição
linhas = []
padrao = re.compile(r"\[(\d+:\d+:\d+) - (\d+:\d+:\d+)\] ?(.*)")
for l in (SAIDA / f"{AUDIO.stem}_tempos.txt").read_text(encoding="utf-8").splitlines():
    m = padrao.match(l)
    if not m:
        continue
    a, b, txt = seg(m[1]), seg(m[2]), m[3]
    b = max(b, a + 1)
    votos = defaultdict(float)
    for t0, t1, spk in turnos:
        if t1 > a and t0 < b:
            votos[NOMES[spk]] += sobrepoe(a, b, t0, t1)
    quem = max(votos, key=votos.get) if votos else "?"
    linhas.append((a, b, quem, txt))

# Intervenções: linhas cujo falante difere de quem conduz o bloco
def dono(t):
    return next(n for b0, b1, n in BLOCOS if b0 <= t < b1)
interv = [(a, quem, txt) for a, b, quem, txt in linhas if quem != dono(a)]
print(f"\n{len(linhas)} linhas; {len(interv)} fora do condutor do bloco")

Victorino      {'Victorino': '81%', 'Silvio': '13%', 'Maria Cláudia': '5%'}
Silvio         {'Silvio': '99%', 'Victorino': '0%', 'Maria Cláudia': '0%'}
Maria Cláudia  {'Maria Cláudia': '90%', 'Silvio': '10%', 'Andrei': '0%'}
Andrei         {'Andrei': '38%', 'Victorino': '29%', 'Silvio': '26%', 'Maria Cláudia': '7%'}

1567 linhas; 378 fora do condutor do bloco


## 8. Salvar transcrição com falantes

In [9]:
# 1) linha a linha, com tempos
(SAIDA / f"{AUDIO.stem}_falantes.txt").write_text(
    "\n".join(f"[{fmt(a)} - {fmt(b)}] {quem}: {txt}" for a, b, quem, txt in linhas),
    encoding="utf-8")

# 2) por turno de fala: junta linhas seguidas do mesmo falante (versão de leitura)
turnos_txt = []
for a, b, quem, txt in linhas:
    if turnos_txt and turnos_txt[-1][2] == quem:
        turnos_txt[-1][1] = b
        turnos_txt[-1][3].append(txt)
    else:
        turnos_txt.append([a, b, quem, [txt]])
(SAIDA / f"{AUDIO.stem}_falantes_turnos.md").write_text(
    "\n\n".join(f"**{quem}** [{fmt(a)} - {fmt(b)}]\n{' '.join(t)}" for a, b, quem, t in turnos_txt),
    encoding="utf-8")

print("Salvo:", f"{AUDIO.stem}_falantes.txt", "e", f"{AUDIO.stem}_falantes_turnos.md")

Salvo: video_falantes.txt e video_falantes_turnos.md
